# CPD — Copy-Paste Detector (part of PMD)

[CPD](https://pmd.github.io/pmd/pmd_userdocs_cpd.html) finds duplicated code across your Java source files.

## What CPD produces
- **Duplicate blocks** with: token count, start line, file, source snippet.
- Formats: `text`, `xml`, `csv`, `vs` (Visual Studio), `csv_with_linecount_per_file`.
- Key tuneable: **`--minimum-tokens`** (default 100) — lower = more matches.

## CLI (PMD 7)
```
pmd cpd --dir <source-dir> --minimum-tokens <N> --language java [--format <fmt>]
```

## 1. Configuration

In [1]:
from __future__ import annotations
import os, subprocess, sys
from pathlib import Path

_base = Path(r"F:\java_metrics")
PROJECT_ROOT   = Path(os.environ.get("JAVA_PROJECT_ROOT", _base / "sample-java-app" / "src")).resolve()
PMD_HOME       = Path(os.environ.get("PMD_HOME", _base / "tools" / "pmd-bin-7.23.0")).resolve()
OUTPUT_DIR     = _base / "cpd_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PMD_EXE        = PMD_HOME / "bin" / ("pmd.bat" if sys.platform == "win32" else "pmd")
MINIMUM_TOKENS = os.environ.get("CPD_MINIMUM_TOKENS", "25")  # low for small projects
FORMATS        = ["text", "xml", "csv"]

assert PMD_EXE.is_file(),     f"pmd not found: {PMD_EXE}"
assert PROJECT_ROOT.is_dir(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"PMD_EXE        : {PMD_EXE}")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"MINIMUM_TOKENS : {MINIMUM_TOKENS}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")

PMD_EXE        : F:\java_metrics\tools\pmd-bin-7.23.0\bin\pmd.bat
PROJECT_ROOT   : F:\java_metrics\sample-java-app\src
MINIMUM_TOKENS : 25
OUTPUT_DIR     : F:\java_metrics\cpd_out


## 2. List Java source files

In [2]:
java_files = sorted(PROJECT_ROOT.rglob("*.java"))
print(f"Java files found: {len(java_files)}")
for f in java_files:
    print(" ", f.relative_to(PROJECT_ROOT))

Java files found: 1
  main\java\com\example\App.java


## 3. Run CPD — raw output (text, xml, csv)

In [3]:
results = {}

for fmt in FORMATS:
    ext = {"text": "txt", "xml": "xml", "csv": "csv"}.get(fmt, fmt)
    report_file = OUTPUT_DIR / f"cpd-report.{ext}"
    cmd = [
        str(PMD_EXE), "cpd",
        "--dir", str(PROJECT_ROOT),
        "--minimum-tokens", MINIMUM_TOKENS,
        "--language", "java",
        "--format", fmt,
        "--report-file", str(report_file),
    ]
    print(f"\n{'='*60}")
    print(f"Running CPD [{fmt}]: {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print("STDOUT:", proc.stdout or "(empty)")
    print("STDERR:", proc.stderr or "(empty)")
    # Exit 4 = duplicates found, 0 = no duplicates, others = error
    print("Exit code:", proc.returncode,
          "(4 = duplicates found, 0 = no duplicates)")
    results[fmt] = report_file


Running CPD [text]: F:\java_metrics\tools\pmd-bin-7.23.0\bin\pmd.bat cpd --dir F:\java_metrics\sample-java-app\src --minimum-tokens 25 --language java --format text --report-file F:\java_metrics\cpd_out\cpd-report.txt


STDOUT: (empty)
STDERR: (empty)
Exit code: 0 (4 = duplicates found, 0 = no duplicates)

Running CPD [xml]: F:\java_metrics\tools\pmd-bin-7.23.0\bin\pmd.bat cpd --dir F:\java_metrics\sample-java-app\src --minimum-tokens 25 --language java --format xml --report-file F:\java_metrics\cpd_out\cpd-report.xml


STDOUT: (empty)
STDERR: (empty)
Exit code: 0 (4 = duplicates found, 0 = no duplicates)

Running CPD [csv]: F:\java_metrics\tools\pmd-bin-7.23.0\bin\pmd.bat cpd --dir F:\java_metrics\sample-java-app\src --minimum-tokens 25 --language java --format csv --report-file F:\java_metrics\cpd_out\cpd-report.csv


STDOUT: (empty)
STDERR: (empty)
Exit code: 0 (4 = duplicates found, 0 = no duplicates)


## 4. Raw text report

In [4]:
txt = results["text"]
if txt.is_file():
    content = txt.read_text(encoding="utf-8", errors="replace")
    print(f"=== {txt.name} ===")
    print(content or "(empty — no duplicates at this threshold)")
else:
    print("Text report not produced.")

=== cpd-report.txt ===
(empty — no duplicates at this threshold)


## 5. Raw XML report

In [5]:
xml = results["xml"]
if xml.is_file():
    print(f"=== {xml.name} ===")
    print(xml.read_text(encoding="utf-8", errors="replace"))
else:
    print("XML report not produced.")

=== cpd-report.xml ===
<?xml version="1.0" encoding="UTF-8"?>
<pmd-cpd xmlns="https://pmd-code.org/schema/cpd-report"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         pmdVersion="7.23.0"
         timestamp="2026-03-28T15:05:40.9303433+05:30"
         version="1.0.0"
         xsi:schemaLocation="https://pmd-code.org/schema/cpd-report https://pmd.github.io/schema/cpd-report_1_0_0.xsd">
   <file path="F:\java_metrics\sample-java-app\src\main\java\com\example\App.java"
         totalNumberOfTokens="25"/>
</pmd-cpd>



## 6. Raw CSV report

In [6]:
csv_f = results["csv"]
if csv_f.is_file():
    print(f"=== {csv_f.name} ===")
    print(csv_f.read_text(encoding="utf-8", errors="replace"))
else:
    print("CSV report not produced.")

=== cpd-report.csv ===
lines,tokens,occurrences



## 7. All output files

In [7]:
print("Files in output directory:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

Files in output directory:
  cpd-report.csv  (26 bytes)
  cpd-report.txt  (0 bytes)
  cpd-report.xml  (517 bytes)
